# Malkus waterwheel: SI-DL summary

This notebook reproduces the data behind the retained Malkus summary table. It simulates the chaotic Malkus waterwheel, constructs the nondimensional output, performs the 10,000-point SI-DL search, evaluates the published continuous IT-PI reference groups, and writes the final table.

All files in `csv/` are generated below. No pickle or precomputed summary file is required. The default notebook uses the `pi` conda environment.

## 1. Imports and experiment settings

The defaults reproduce the retained result: 100 parameter sets, 500 time samples per integration, a deterministic 10,000-point subset, Gaussian-kernel SI-DL with mirror boundaries, and differential evolution initialized near the published IT-PI solution.

The full SI-DL search takes roughly 15–20 minutes on the machine used to create the retained table. Environment variables can reduce the settings for a smoke test without changing the notebook code.

In [ ]:
from __future__ import annotations

import os
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution

WORK_DIR = Path.cwd()
if not (WORK_DIR / "malkus.ipynb").exists() and (WORK_DIR / "Malkus" / "malkus.ipynb").exists():
    WORK_DIR = WORK_DIR / "Malkus"

CSV_DIR = WORK_DIR / "csv"
FIG_DIR = WORK_DIR / "figures"
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = WORK_DIR.parent
for module_dir in (PROJECT_ROOT / "Compare" / "MI-DL", PROJECT_ROOT / "SI-DL-main"):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import midl
import SI_DL

RANDOM_STATE = 42
NUM_PARAMETER_SETS = int(os.environ.get("MALKUS_NUM_PARAMETER_SETS", "100"))
TIME_POINTS = int(os.environ.get("MALKUS_TIME_POINTS", "500"))
SINGLE_SET_SAMPLE_SIZE = int(os.environ.get("MALKUS_SAMPLE_SIZE", "10000"))

MI_K_NEIGHBORS = 6
SI_BANDWIDTH = 0.1
SI_MAXITER = int(os.environ.get("MALKUS_SI_MAXITER", "40"))
SI_POPSIZE = int(os.environ.get("MALKUS_SI_POPSIZE", "8"))
SI_INIT_JITTER_SCALE = 0.10

VARIABLE_LABELS = ["r", "q_1", "I", "nu", "K", "g", "omega", "a_1", "b_1"]
D_IN = np.array([
    [1, 0, 2, 2, 0, 1, 0, 0, 0],
    [0, -1, 0, -1, -1, -2, -1, 0, 0],
    [0, 1, 1, 1, 0, 0, 0, 1, 1],
], dtype=float)

## 2. Simulate the Malkus waterwheel

For each sampled physical parameter set, the three-state nonlinear waterwheel system is integrated from 0 to 50 seconds. The dimensional acceleration is then converted to the nondimensional output

\[
\Pi_0=\frac{d\omega/dt}{K^2}.
\]

As in the original Malkus example, every input column is divided by its maximum before nonnegative state rows are retained. With the default settings this produces 28,979 valid samples.

In [ ]:
def chaotic_wheel_system(t, state, r, q_1, inertia, nu, leakage):
    omega, a_1, b_1 = state
    return [
        (np.pi * r * 9.81 / inertia) * a_1 - (nu / inertia) * omega,
        omega * b_1 - leakage * a_1,
        q_1 - leakage * b_1 - omega * a_1,
    ]

np.random.seed(RANDOM_STATE)
r_values = np.random.uniform(0.3, 0.7, NUM_PARAMETER_SETS)
q1_values = np.random.uniform(0.0001, 0.0005, NUM_PARAMETER_SETS)
inertia_values = np.random.uniform(0.05, 0.2, NUM_PARAMETER_SETS)
nu_values = np.random.uniform(0.01, 0.1, NUM_PARAMETER_SETS)
leakage_values = np.random.uniform(0.01, 0.1, NUM_PARAMETER_SETS)

t_eval = np.linspace(0.0, 50.0, TIME_POINTS)
initial_state = [0.3, 0.00001, 0.00001]
input_rows = []
output_rows = []

for r, q_1, inertia, nu, leakage in zip(
    r_values, q1_values, inertia_values, nu_values, leakage_values
):
    solution = solve_ivp(
        chaotic_wheel_system,
        (0.0, 50.0),
        initial_state,
        t_eval=t_eval,
        args=(r, q_1, inertia, nu, leakage),
    )
    for omega, a_1, b_1 in zip(*solution.y):
        input_rows.append([r, q_1, inertia, nu, leakage, 9.81, omega, a_1, b_1])
        domega_dt = (np.pi * r * 9.81 / inertia) * a_1 - (nu / inertia) * omega
        output_rows.append(domega_dt / leakage**2)

raw_inputs = np.asarray(input_rows, dtype=float)
raw_output = np.asarray(output_rows, dtype=float)
normalized_inputs = raw_inputs / raw_inputs.max(axis=0)
valid = np.all(normalized_inputs >= 0.0, axis=1) & np.isfinite(raw_output)
X = normalized_inputs[valid]
Y = raw_output[valid]

data = pd.DataFrame(X, columns=VARIABLE_LABELS)
data["Pi0"] = Y
data.to_csv(CSV_DIR / "data.csv", index=False)

print(f"Raw samples: {len(raw_inputs):,}")
print(f"Valid samples: {len(data):,}")
print(f"Wrote {CSV_DIR / 'data.csv'}")
data.head()

## 3. Dimensionless groups and reference solution

The dimensional null space contains six basis directions. The sign convention matches the original Malkus IT-PI notebook.

`ITPI_OMEGAS` is the continuous two-group solution reported by that original optimization. It is included as a declared reference, analogous to a known/reference group in the Colebrook notebook. Its MI and SI-DL scores are not stored: they are recalculated below on the newly simulated 10,000-point subset.

In [ ]:
ITPI_OMEGAS = np.array([
    [-0.03, 0.00, -0.50, 0.54, -1.00, -0.05, 0.55, -0.04, 0.00],
    [ 0.60, 0.00, -0.50,-0.09, -1.00,  0.58,-0.07,  0.58, 0.00],
], dtype=float)

def malkus_basis():
    basis = np.asarray(
        SI_DL.calc_basis(D_IN, D_IN.shape[1] - np.linalg.matrix_rank(D_IN)),
        dtype=float,
    )
    basis[3:6, :] = -basis[3:6, :]
    return basis.T

basis = malkus_basis()
print("Basis shape:", basis.shape)

## 4. Scoring and exponent utilities

The two learned coordinates are evaluated in logarithmic form before exponentiation. Mutual information uses the project MI-DL estimator with `k=6`. SI-DL uses the covariance score, bandwidth 0.1, and a mirror boundary correction.

In [ ]:
def scale_exponents(exponents, canonical_sign=True):
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))
    if scale <= 1e-12:
        return row
    scaled = row / scale
    if canonical_sign:
        first = np.flatnonzero(np.abs(scaled) > 1e-10)
        if first.size and scaled[first[0]] < 0.0:
            scaled = -scaled
    return scaled


def params_to_omegas(params, basis_matrix, canonical_sign=False):
    coefficients = np.asarray(params, dtype=float).reshape(2, basis_matrix.shape[1])
    raw = coefficients @ basis_matrix.T
    return np.asarray([
        scale_exponents(row, canonical_sign=canonical_sign) for row in raw
    ])


def params_from_omegas(omegas, basis_matrix, canonical_sign=False):
    rows = []
    for row in np.asarray(omegas, dtype=float):
        coefficients, *_ = np.linalg.lstsq(
            basis_matrix,
            scale_exponents(row, canonical_sign=canonical_sign),
            rcond=None,
        )
        rows.append(coefficients)
    return np.asarray(rows).reshape(-1)


def pi_from_omegas(inputs, omegas):
    log_inputs = np.log(np.asarray(inputs, dtype=float))
    omega_array = np.asarray(omegas, dtype=float)
    log_pi = np.sum(log_inputs[:, None, :] * omega_array[None, :, :], axis=2)
    return np.exp(log_pi)


def formula_from_exponents(omegas, decimals=3):
    lines = []
    for index, row in enumerate(np.asarray(omegas, dtype=float), start=1):
        terms = [
            f"{label}^{value:.{decimals}f}"
            for label, value in zip(VARIABLE_LABELS, row)
            if abs(float(value)) >= 5e-4
        ]
        lines.append(f"pi{index}: " + " * ".join(terms))
    return "\n".join(lines)

In [ ]:
def score_sidl(feature, response):
    score = SI_DL.explained_variance_score(
        feature,
        response,
        bandwidth=SI_BANDWIDTH,
        boundary="mirror",
    )
    fitted = np.asarray(score["m_hat"], dtype=float).reshape(-1)
    residual = np.asarray(response, dtype=float) - fitted
    return {
        "S_cov": float(score["S_cov"]),
        "S_cov_raw": float(score["S_cov_raw"]),
        "sidl_error_clipped": float(1.0 - score["S_cov"]),
        "sidl_error_raw": float(1.0 - score["S_cov_raw"]),
        "sidl_bandwidth": float(score["bandwidth"]),
        "sidl_n_retained": int(score["n_retained"]),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "nrmse": float(np.sqrt(np.mean(residual**2)) / np.std(response, ddof=0)),
        "corr_Y_mhat": float(np.corrcoef(response, fitted)[0, 1]),
        "max_abs_residual": float(np.max(np.abs(residual))),
        "median_abs_residual": float(np.median(np.abs(residual))),
    }


def information_metrics(feature, response):
    information = midl.information_lower_bound(
        np.asarray(feature, dtype=float),
        np.asarray(response, dtype=float),
        k_neighbors=MI_K_NEIGHBORS,
        random_state=RANDOM_STATE,
    )
    return {
        "mutual_information": float(information["mutual_information"]),
        "epsilon_lb_normalized": float(
            information["epsilon_lb"] / np.var(response, ddof=0)
        ),
    }

## 5. Select the deterministic 10,000-point set

The same subset is used for SI-DL optimization, final SI-DL metrics, and the IT-PI reference metrics. This avoids mixing search and reporting populations in the retained comparison.

In [ ]:
subset_size = min(SINGLE_SET_SAMPLE_SIZE, len(X))
rng = np.random.default_rng(RANDOM_STATE + 1)
subset_index = (
    np.arange(len(X))
    if len(X) <= subset_size
    else np.sort(rng.choice(len(X), size=subset_size, replace=False))
)
X_eval = X[subset_index]
Y_eval = Y[subset_index]

print(f"Selected {len(X_eval):,} points for search and evaluation")

## 6. Run the SI-DL differential-evolution search

The optimizer searches two independent dimensionless groups. Its initial population is centered near the continuous IT-PI reference and jittered deterministically. Progress is written to `csv/de_log.csv` after each iteration.

In [ ]:
parameter_count = 2 * basis.shape[1]
population_size = max(5, SI_POPSIZE * parameter_count)
center = params_from_omegas(ITPI_OMEGAS, basis, canonical_sign=False)
rng = np.random.default_rng(RANDOM_STATE)
initial_population = center + rng.normal(
    0.0,
    SI_INIT_JITTER_SCALE,
    size=(population_size, parameter_count),
)
initial_population[0] = center
initial_population = np.clip(initial_population, -2.0, 2.0)

log_rows = []
evaluation_count = 0
best_seen = -np.inf
search_started = time.time()


def objective(params):
    global evaluation_count, best_seen
    evaluation_count += 1
    try:
        omegas = params_to_omegas(params, basis, canonical_sign=False)
        if np.linalg.matrix_rank(omegas, tol=1e-8) < 2:
            return 1e6
        value = score_sidl(pi_from_omegas(X_eval, omegas), Y_eval)["S_cov"]
    except Exception:
        return 1e6
    if not np.isfinite(value):
        return 1e6
    best_seen = max(best_seen, float(value))
    return -float(value)


def record_progress(*args, **kwargs):
    intermediate = kwargs.get("intermediate_result")
    if intermediate is None and args and hasattr(args[0], "fun"):
        intermediate = args[0]
    best_score = -float(intermediate.fun) if intermediate is not None else best_seen
    log_rows.append({
        "iteration": len(log_rows) + 1,
        "elapsed_seconds": float(time.time() - search_started),
        "objective_evaluations": int(evaluation_count),
        "best_S_cov": float(best_score),
    })
    pd.DataFrame(log_rows).to_csv(CSV_DIR / "de_log.csv", index=False)
    print(
        f"iteration {len(log_rows):03d}: S_cov={best_score:.6f}, "
        f"evaluations={evaluation_count}, elapsed={(time.time()-search_started)/60:.2f} min",
        flush=True,
    )
    return False


optimizer_result = differential_evolution(
    objective,
    bounds=[(-2.0, 2.0)] * parameter_count,
    maxiter=SI_MAXITER,
    popsize=SI_POPSIZE,
    seed=RANDOM_STATE,
    init=initial_population,
    polish=False,
    updating="immediate",
    workers=1,
    callback=record_progress,
)

sidl_omegas = params_to_omegas(
    optimizer_result.x, basis, canonical_sign=False
)
pd.DataFrame(log_rows).to_csv(CSV_DIR / "de_log.csv", index=False)

## 7. Recalculate the summary data

Both rows are scored on the freshly generated 10,000-point set. `summary.csv` contains all numerical columns displayed or audited, while `exponents.csv` stores each exponent separately.

In [ ]:
methods = {
    "SI-DL single-set DE": sidl_omegas,
    "IT-PI continuous": ITPI_OMEGAS,
}
summary_rows = []
exponent_rows = []

for method, omegas in methods.items():
    feature = pi_from_omegas(X_eval, omegas)
    sidl_metrics = score_sidl(feature, Y_eval)
    mi_metrics = information_metrics(feature, Y_eval)
    summary_rows.append({
        "method": method,
        "n_single_set_samples": int(len(X_eval)),
        "selection_set": "same_10000_points",
        "selection_metric": "max_S_cov_clipped",
        "boundary": "mirror",
        **sidl_metrics,
        **mi_metrics,
        "optimizer_fun": float(optimizer_result.fun) if method.startswith("SI-DL") else np.nan,
        "optimizer_success": bool(optimizer_result.success) if method.startswith("SI-DL") else np.nan,
        "optimizer_message": str(optimizer_result.message) if method.startswith("SI-DL") else "",
        "objective_evaluations": int(evaluation_count) if method.startswith("SI-DL") else np.nan,
        "elapsed_seconds": float(time.time() - search_started) if method.startswith("SI-DL") else np.nan,
        "formula": formula_from_exponents(omegas),
    })
    for pi_index, row in enumerate(omegas, start=1):
        for label, exponent in zip(VARIABLE_LABELS, row):
            exponent_rows.append({
                "method": method,
                "pi_group": f"pi{pi_index}",
                "variable": label,
                "normalized_exponent": float(exponent),
            })

summary = pd.DataFrame(summary_rows)
exponents = pd.DataFrame(exponent_rows)
summary.to_csv(CSV_DIR / "summary.csv", index=False)
exponents.to_csv(CSV_DIR / "exponents.csv", index=False)

print(f"Wrote {CSV_DIR / 'summary.csv'}")
print(f"Wrote {CSV_DIR / 'exponents.csv'}")
summary[["method", "S_cov", "sidl_error_clipped", "nrmse", "mutual_information"]]

## 8. Generate the summary figure

The table is built only from the newly written `summary.csv`. Formula text is wrapped so both dimensionless groups remain fully visible.

In [ ]:
summary = pd.read_csv(CSV_DIR / "summary.csv")

def wrap_formula(formula):
    lines = []
    for line in str(formula).splitlines():
        lines.extend(textwrap.wrap(line, width=74, subsequent_indent="  "))
    return "\n".join(lines)

rows = []
for _, row in summary.iterrows():
    rows.append([
        row["method"],
        f"{row['S_cov']:.6f}",
        f"{row['S_cov_raw']:.6f}",
        f"{row['sidl_error_clipped']:.6f}",
        f"{row['nrmse']:.6f}",
        f"{row['corr_Y_mhat']:.6f}",
        f"{row['mutual_information']:.6f}",
        wrap_formula(row["formula"]),
    ])

fig, ax = plt.subplots(figsize=(18.5, 5.0), dpi=240)
ax.axis("off")
ax.set_title(
    "Malkus SI-DL, 10,000 points, boundary=mirror, bandwidth=0.1",
    fontsize=17,
    weight="bold",
    pad=18,
)
table = ax.table(
    cellText=rows,
    colLabels=["Method", "S_cov", "S_cov_raw", "1-S_cov", "NRMSE", "corr", "MI", "Pi groups"],
    cellLoc="center",
    colLoc="center",
    colWidths=[0.12, 0.07, 0.08, 0.075, 0.075, 0.07, 0.07, 0.44],
    bbox=[0.015, 0.04, 0.97, 0.80],
)
table.auto_set_font_size(False)
for (row, column), cell in table.get_celld().items():
    cell.set_edgecolor("#404040")
    cell.set_linewidth(0.8)
    cell.PAD = 0.02
    if row == 0:
        cell.set_facecolor("#1f2937")
        cell.get_text().set_color("white")
        cell.get_text().set_weight("bold")
        cell.get_text().set_fontsize(9.0)
    else:
        cell.set_facecolor("#f8fafc" if row % 2 else "#ffffff")
        cell.get_text().set_fontsize(7.1 if column == 7 else 8.6)
        if column == 7:
            cell.get_text().set_ha("left")
        if column == 0:
            cell.get_text().set_weight("bold")

fig.savefig(FIG_DIR / "summary.png", bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Wrote {FIG_DIR / 'summary.png'}")

## Outputs

A full run generates:

- `csv/data.csv`
- `csv/de_log.csv`
- `csv/summary.csv`
- `csv/exponents.csv`
- `figures/summary.png`